# Game Team Stats to Delta table

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ACCOUNT    = "footballmanagerli"
SRC        = f"abfss://alsource@{ACCOUNT}.dfs.core.windows.net/game-team-stats/"
DEST       = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/game-team-stats/game_team_stats"
CHECKPOINT = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/_checkpoints/game_team_stats"
TABLE      = "nfl.game_team_stats"

In [ ]:
# Use minimal schema for base fields; stats are dynamic
base_schema = StructType([
    StructField("teamId",      StringType()),
    StructField("teamName",    StringType()),
    StructField("teamAbbr",    StringType()),
])

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS nfl")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    gameId        STRING    COMMENT 'Game ID (part of composite key)',
    teamId        STRING    COMMENT 'Team ID (part of composite key)',
    teamName      STRING,
    teamAbbr      STRING,
    stats         MAP<STRING, STRING> COMMENT 'Dynamic ESPN boxscore stats (passing yards, rushing, etc)',
    snapshot_at   TIMESTAMP COMMENT 'When the snapshot file was written',
    source_file   STRING,
    ingested_at   TIMESTAMP
)
USING DELTA
LOCATION '{DEST}'
""")

In [ ]:
# Read JSON as untyped to capture dynamic stats
raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .load(SRC)
    .select(
        "*",
        F.col("_metadata.file_path").alias("source_file"),
        F.col("_metadata.file_modification_time").alias("snapshot_at"),
    ))

# Extract known fields and capture dynamic stats as a map
game_stats = (raw
    .select(
        F.col("gameId").cast(StringType()),
        F.col("teamId"),
        F.col("teamName"),
        F.col("teamAbbr"),
        # Convert entire row to JSON then extract all fields as a map of strings
        F.from_json(F.to_json(F.struct("*")), "MAP<STRING, STRING>").alias("all_fields"),
        "snapshot_at",
        "source_file",
        F.current_timestamp().alias("ingested_at"),
    )
    .select(
        F.col("gameId"),
        F.col("teamId"),
        F.col("teamName"),
        F.col("teamAbbr"),
        # Drop known fields from the map, keep the rest as stats
        F.map_filter(
            F.col("all_fields"),
            lambda k, v: ~F.isin(k, ["gameId", "teamId", "teamName", "teamAbbr", "source_file", "snapshot_at", "ingested_at"])
        ).alias("stats"),
        "snapshot_at",
        "source_file",
        "ingested_at",
    ))

In [ ]:
def upsert_batch(batch_df, batch_id):
    # Dedupe within the batch: keep the newest snapshot per game/team combo
    w = Window.partitionBy("gameId", "teamId").orderBy(F.col("snapshot_at").desc())
    latest = (batch_df
        .withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn"))

    # Merge into the table: insert new records, update if snapshot is newer
    (DeltaTable.forName(spark, TABLE).alias("t")
        .merge(latest.alias("s"), "t.gameId = s.gameId AND t.teamId = s.teamId")
        .whenMatchedUpdateAll(condition="s.snapshot_at > t.snapshot_at")
        .whenNotMatchedInsertAll()
        .execute())

(game_stats.writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
    .awaitTermination())

In [ ]:
%sql
SELECT COUNT(*) AS rows FROM nfl.game_team_stats;

SELECT COUNT(DISTINCT gameId) AS unique_games FROM nfl.game_team_stats;

SELECT gameId, teamId, teamName, stats FROM nfl.game_team_stats ORDER BY gameId DESC LIMIT 10;